In [37]:
print("Hola, este es mi primer chatbot, By Ing. Luis Uribe")
%pip -q install nltk
import sys
import nltk
print("Python:", sys.version.split()[0])
print("NLTK:", nltk.__version__)
assert sys.version_info >= (3, 10)

import unicodedata
from nltk.tokenize import RegexpTokenizer
tokenizador = RegexpTokenizer(r"\w+")

# --- Función de Normalización ---
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(
        letra for letra in texto
        if unicodedata.category(letra) != "Mn"
    )
    palabras = tokenizador.tokenize(texto)
    return " ".join(palabras)

# --- Bloque Inicial de Intenciones Técnicas (Maquinas Nordson Asymtek SL-940) ---
intenciones = {
    "saludo": {
        "reglas": ["hola", "buenos dias", "buenas tardes", "iniciar", "ayuda"],
        "respuesta": "Hola. Soy tu asistente de mantenimiento para las maquinas Nordson Asymtek SL-940. ¿Qué falla o síntoma presenta el equipo? (goteo, boquilla tapada, burbujas, software, presion)"
    },
    "falla_goteo": {
        "reglas": ["gotea", "goteo", "goteando", "cae material", "valvula gotea"],
        "respuesta": "[Causa Raíz - Goteo]: 1. Aire atrapado en la cámara del fluido (requiere purga). 2. Presión de aire de asistencia insuficiente. 3. Desgaste en la aguja o asiento de la valvula (SC-400/SC-280)."
    },
    "falla_bloqueo": {
        "reglas": ["tapado", "bloqueado", "obstruido", "no sale material", "boquilla tapada", "no sale nada"],
        "respuesta": "[Causa Raíz - Obstrucción]: 1. Curado prematuro del fluido en la punta (limpiar con solvente). 2. Temperatura del calefactor muy baja. 3. Presión del tanque en cero o baja."
    },
    "falla_burbujas": {
        "reglas": ["burbujas", "aire en fluido", "poros", "espuma", "huecos", "burbuja"],
        "respuesta": "[Causa Raíz - Burbujas]: 1. Exceso de presión en el tanque de fluido (disuelve aire). 2. El material fue agitado y no se dejó desgasificar. 3. Entrada de aire por acoples rápidos dañados."
    }
}
print("Intenciones cargadas:", len(intenciones))

# --- Bloque de Actualización (Update) ---
intenciones.update({
    "easycoat_software": {
        "reglas": ["software", "easycoat", "error software", "no responde programa", "ruta", "error"],
        "respuesta": "[Software EasyCoat]: Verifica la comunicación de la tarjeta principal, reinicia el software y valida que el archivo de configuracion(.ini) esté cargado correctamente."
    },
    "presion_tanque": {
        "reglas": ["presion", "presion baja", "baja presion", "tanque sin presion", "regulador", "presion en el tanque"],
        "respuesta": "[Línea de Fluido]: Valida el manómetro del tanque del conformal coating. Revisa que el regulador principal reciba al menos 80 psi constantes desde la línea de planta."
    },
    "mantenimiento_limpieza": {
        "reglas": ["como limpiar", "limpieza modulo", "solvente", "limpio la boquilla"],
        "respuesta": "[Mantenimiento Preventivo]: Utiliza el solvente aprobado por el fabricante para tu tipo de conformal. Remoja la boquilla desmontada por 10 minutos y usa un cepillo de cerdas suaves."
    },
    "falla_espesor": {
        "reglas": ["espesor", "capa delgada", "mucha capa", "espesor incorrecto"],
        "respuesta": "[Causa Raíz - Parámetros de Capa]: 1. Velocidad de avance incorrecta en EasyCoat. 2. Viscosidad del conformal fuera de especificación (mide con copa Ford #2). 3. Presión de atomización descalibrada."
    },
    "falla_laser": {
        "reglas": ["laser", "sensor de altura", "no mide altura", "calibracion z"],
        "respuesta": "[Sensor Láser]: 1. Lente del sensor óptico sucio con residuos de conformal (limpiar con hisopo seco). 2. Placa de calibración desalineada."
    },
    "despedida": {
        "reglas": ["adios", "hasta luego", "gracias", "solucionado", "listo"],
        "respuesta": "Excelente. Asegúrate de documentar correctamente el ticket en el sistema Digital Factory. ¡Buen turno!"
    }
})
print("Intenciones cargadas con actualizaciones:", len(intenciones))

# --- Función de Respuesta ---
def responder(mensaje):
    texto = normalizar(mensaje)
    if texto == "salir":
        return "salir", "Sesion finalizada."

    coincidencias = []
    for nombre, datos in intenciones.items():
        for regla in datos["reglas"]:
            patron = " " + normalizar(regla) + " "
            if patron in " " + texto + " ":
                coincidencias.append(nombre)
                break

    if len(coincidencias) == 1:
        nombre = coincidencias[0]
        return nombre, intenciones[nombre]["respuesta"]

    if len(coincidencias) > 1:
        return "ambigua", "Por favor, consulta un solo síntoma a la vez para evitar confusiones de diagnóstico."

    return "desconocida", "Síntoma no identificado. Intenta describirlo con palabras clave como: goteo, tapado, burbujas o presion."


# --- Bucle de Consola Activo para Interacción Real ---
print("\nAsistente de Procesos para maquinas Nordson Asymtek SL-940. Escribe salir para cerrar.")
while True:
    mensaje = input("Tu: ")
    intencion, respuesta = responder(mensaje)
    print("Intencion:", intencion)
    print("Bot:", respuesta)
    if intencion == "salir":
        break


# --- Matriz de Pruebas Automatizadas Adaptada a fallas de maquina ---
pruebas = [
    ("Hola", "saludo"),
    ("La boquilla está goteando material", "falla_goteo"),
    ("¿Cómo limpio la boquilla?", "mantenimiento_limpieza"),
    ("No sale nada, está obstruido", "falla_bloqueo"),
    ("Tengo baja presión en el tanque", "presion_tanque"),
    ("presion", "presion_tanque"),
    ("El programa EasyCoat arrojó un error", "easycoat_software"),
    ("software", "easycoat_software"),
    ("¿Tiene burbujas el circuito?", "falla_burbujas"),
    ("Gracias, quedó solucionado", "despedida"),
    ("La máquina hace un ruido en el motor", "desconocida"),
    ("Tengo goteo y burbujas al mismo tiempo", "ambigua"),
    ("", "desconocida"),
    ("salir", "salir")
]

aciertos = 0
print("\n=== INICIANDO VALIDACIÓN AUTOMÁTICA DE MATRIZ DE PRUEBAS ===")

for mensaje, esperada in pruebas:
    detectada, respuesta = responder(mensaje)
    correcto = detectada == esperada
    aciertos += int(correcto)
    print(f"\nEntrada: {repr(mensaje)}")
    print(f"Esperada: {esperada} | Detectada: {detectada}")
    print(f"Bot: {respuesta} | Cumple: {correcto}")

print("\n=======================================================")
print(f"Resultado Final - Coincidencias Exitosas: {aciertos} / {len(pruebas)}")
print("=======================================================")




Hola, este es mi primer chatbot, By Ing. Luis Uribe
Python: 3.13.16
NLTK: 3.9.1
Intenciones cargadas: 4
Intenciones cargadas con actualizaciones: 10

Asistente de Procesos para maquinas Nordson Asymtek SL-940. Escribe salir para cerrar.
Tu: 
Intencion: desconocida
Bot: Síntoma no identificado. Intenta describirlo con palabras clave como: goteo, tapado, burbujas o presion.
Tu: gracias
Intencion: despedida
Bot: Excelente. Asegúrate de documentar correctamente el ticket en el sistema Digital Factory. ¡Buen turno!
Tu: salir
Intencion: salir
Bot: Sesion finalizada.

=== INICIANDO VALIDACIÓN AUTOMÁTICA DE MATRIZ DE PRUEBAS ===

Entrada: 'Hola'
Esperada: saludo | Detectada: saludo
Bot: Hola. Soy tu asistente de mantenimiento para las maquinas Nordson Asymtek SL-940. ¿Qué falla o síntoma presenta el equipo? (goteo, boquilla tapada, burbujas, software, presion) | Cumple: True

Entrada: 'La boquilla está goteando material'
Esperada: falla_goteo | Detectada: falla_goteo
Bot: [Causa Raíz - Goteo]: